# TRL SFT Training V2 — with Multi-Agent Rollouts

**Same structure as V1 training, now includes multi-agent rollouts where teacher uses REQUEST_REVIEW + REVISE_SECTION.**

## Pipeline
1. Teacher (Llama 8B) plays episodes — mix of single-agent + multi-agent
2. Rejection sampling (reward >= 0.50)
3. TRL SFT training on Qwen 2.5-0.5B
4. Before/after evaluation

**Runtime:** ~45-60 min on T4.

## Cell 1 — Install dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q numpy==1.26.4
!pip install -q transformers==4.44.0 trl==0.11.4 datasets==2.21.0 accelerate==0.33.0 peft==0.12.0 bitsandbytes==0.43.3 openai requests matplotlib pandas

print('Dependencies installed.')
print('RESTART RUNTIME now: Runtime -> Restart session, then run Cell 2.')

## Cell 2 — Setup + config

**IMPORTANT:** Paste your Groq API key in GROQ_API_KEY below before running.

In [ ]:
import os, json, time, random, re
from pathlib import Path
import requests
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

HF_SPACE_URL      = 'https://jeevan2717-incident-postmortem-writer.hf.space'
GROQ_API_KEY      = ''   # <-- PASTE YOUR GROQ KEY HERE
GROQ_MODEL        = 'llama-3.1-8b-instant'
GROQ_BASE_URL     = 'https://api.groq.com/openai/v1'

STUDENT_MODEL     = 'Qwen/Qwen2.5-0.5B-Instruct'
REWARD_THRESHOLD  = 0.50
N_ROLLOUTS_PER_DIFFICULTY = 10
MULTI_AGENT_PROB  = 0.5

# Output goes to /content/ (Colab local). Download via the file browser when done.
OUTPUT_DIR        = '/content/postmortem_trained_model_v2'
os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(42)

print(f'HF Space:     {HF_SPACE_URL}')
print(f'Student:      {STUDENT_MODEL}')
print(f'Output dir:   {OUTPUT_DIR}')
print(f'Rollouts/difficulty: {N_ROLLOUTS_PER_DIFFICULTY}')
print(f'Multi-agent episodes: {int(MULTI_AGENT_PROB*100)}%')

assert GROQ_API_KEY.strip(), 'Paste your Groq API key in GROQ_API_KEY before running.'
print('Config ready.')


## Cell 3 — Environment + Groq clients

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=GROQ_API_KEY, base_url=GROQ_BASE_URL)

class PostMortemEnv:
    def __init__(self, base_url):
        self.base_url = base_url.rstrip('/')
        self._session = requests.Session()
    def reset(self, difficulty='easy'):
        r = self._session.post(f'{self.base_url}/reset', json={'difficulty': difficulty}, timeout=30)
        r.raise_for_status(); return r.json()
    def step(self, action):
        r = self._session.post(f'{self.base_url}/step', json=action, timeout=30)
        r.raise_for_status(); return r.json()
    def health(self):
        try: return self._session.get(f'{self.base_url}/health', timeout=5).status_code == 200
        except: return False

env = PostMortemEnv(HF_SPACE_URL)
assert env.health(), f'HF Space at {HF_SPACE_URL} not reachable'
print(f'HF Space healthy: {HF_SPACE_URL}')

def call_llm(system, user, temperature=0.0, max_tokens=1500):
    try:
        r = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{'role':'system','content':system},{'role':'user','content':user}],
            temperature=temperature, max_tokens=max_tokens)
        return (r.choices[0].message.content or '').strip()
    except Exception as e:
        print(f'  [LLM error] {e}')
        return ''

def extract_json(text):
    try: return json.loads(text.strip())
    except: pass
    m = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    if m:
        try: return json.loads(m.group(1))
        except: pass
    m = re.search(r'\{.*?\}', text, re.DOTALL)
    if m:
        try: return json.loads(m.group(0))
        except: pass
    return None

print('Env client + LLM client ready.')

## Cell 4 — Agent prompts

In [ ]:
QUERY_SYSTEM = '''You are an expert SRE. Given incident alerts and Slack messages, identify the best service and time window to query.
Respond with ONLY valid JSON: {"service": "<service_name>", "from": "<HH:MM>", "to": "<HH:MM>"}

STRATEGY:
1. Look for DEPLOYMENT or CONFIG CHANGE in Slack (deploy, TTL, migration, release, config, schema).
2. If no deployment, identify which service changed behavior FIRST.
3. Pick a 5-8 minute window AROUND the deployment or first change time.
4. NEVER query the most-alerted service - it is usually a victim.'''

WRITE_SYSTEM = '''You are an expert SRE writing one section of an incident post-mortem.
Write ONLY the section content - no JSON, no section labels, just plain text.
Be specific. Use exact service names and timestamps.'''

REVISE_SYSTEM = '''You are an expert SRE revising a section based on a reviewer critique.
Write ONLY the revised section content. Must be substantially different (30+ chars changed).
Must directly address the critique.'''

SECTION_PROMPTS = {
    'summary':      'Write 2-3 sentences summarizing the incident. MUST name the affected service.',
    'timeline':     'Write a chronological timeline with 5+ timestamped events "HH:MM - what happened".',
    'root_cause':   'Write root cause. MUST name: (1) which service failed, (2) mechanism (deploy/config/leak/credential), (3) specific details.',
    'impact':       'Write impact assessment 30+ words. Include: services affected, duration, users, business impact.',
    'action_items': 'Write 3 numbered action items with owner + due date. Example: "1. Fix X - Owner: team - Due: 2024-08-01"',
}

print('Prompts configured.')

## Cell 5 — Rollout functions (single-agent + multi-agent)

In [ ]:
def build_context(obs, logs):
    alerts = '\n'.join(f"[{a['timestamp']}] [{a['severity']}] {a['service']}: {a['message']}" for a in obs.get('alerts',[]))
    slack  = '\n'.join(f"[{m['timestamp']}] {m['author']}: {m['text']}" for m in obs.get('slack_thread',[]))
    logstr = '\n'.join(f"[{l['timestamp']}] [{l['severity']}] {l['service']}: {l['message']}" for l in (logs or []))
    return f"INCIDENT: {obs.get('incident_title','')}\nALERTS:\n{alerts}\n\nSLACK:\n{slack}\n\nLOGS:\n{logstr or '(none)'}"

def fallback_section(section, obs):
    services = list({a['service'] for a in obs.get('alerts',[])})
    svc = services[0] if services else 'payments'
    return {
        'summary':      f'The {svc} service experienced a significant incident affecting production.',
        'timeline':     '00:00 - First alert\n00:15 - On-call engaged\n00:30 - Service recovered',
        'root_cause':   f'Root cause: {svc} service deployment bug or config error caused degradation.',
        'impact':       f'{svc} unavailable ~30 min. Users experienced errors and timeouts. Revenue impact.',
        'action_items': '1. Add monitoring - Owner: sre - Due: 2024-08-15\n2. Review deploy - Owner: platform - Due: 2024-08-15\n3. Post-mortem review - Owner: sre - Due: 2024-08-15',
    }.get(section, f'Analysis of {svc} incident.')


def run_single_agent_episode(env, difficulty):
    pairs = []
    result = env.reset(difficulty=difficulty)
    obs = result['observation']
    alerts = obs.get('alerts',[])
    services = list({a['service'] for a in alerts})
    slack = obs.get('slack_thread',[])

    query_user = f"INCIDENT: {obs.get('incident_title','')}\nALERTS:\n" + '\n'.join(f"[{a['timestamp']}] {a['service']}: {a['message']}" for a in alerts) + f"\nSLACK:\n" + '\n'.join(f"[{m['timestamp']}] {m['author']}: {m['text']}" for m in slack) + f"\nServices: {services}\nWhich to query?"
    qresp = call_llm(QUERY_SYSTEM, query_user)
    q = extract_json(qresp)
    result = env.step({'action_type':'QUERY_LOGS','query_service':(q or {}).get('service', services[0] if services else 'payments'),'query_from':(q or {}).get('from','00:00'),'query_to':(q or {}).get('to','23:59')})
    obs = result['observation']; logs = obs.get('retrieved_logs') or []
    pairs.append({'prompt_system':QUERY_SYSTEM,'prompt_user':query_user,'completion':qresp,'reward':float(result.get('reward',{}).get('total',0) or 0),'phase':'query'})

    ctx = build_context(obs, logs)
    for section in ['summary','timeline','root_cause','impact','action_items']:
        wuser = f"{ctx}\n\nWRITE THE '{section.upper()}' SECTION:\n{SECTION_PROMPTS[section]}\n\nContent:"
        resp = call_llm(WRITE_SYSTEM, wuser)
        content = resp if resp and len(resp) >= 20 and not resp.startswith('{') else fallback_section(section, obs)
        result = env.step({'action_type':'WRITE_SECTION','section_name':section,'section_content':content})
        pairs.append({'prompt_system':WRITE_SYSTEM,'prompt_user':wuser,'completion':content,'reward':float(result.get('reward',{}).get('total',0) or 0),'phase':f'write_{section}'})
        obs = result['observation']

    authors = [m['author'] for m in slack if m.get('author') != 'pagerduty-bot']
    owner = authors[0] if authors else 'sre'
    env.step({'action_type':'ASSIGN_ACTION_ITEM','action_item_description':f'Prevent recurrence in {services[0] if services else "payments"}','action_item_owner':owner,'action_item_due_date':'2024-08-15'})
    result = env.step({'action_type':'SUBMIT'})
    return pairs, result.get('info',{}).get('grade',{}).get('total_score',0)


def run_multi_agent_episode(env, difficulty):
    pairs = []
    result = env.reset(difficulty=difficulty)
    obs = result['observation']
    alerts = obs.get('alerts',[])
    services = list({a['service'] for a in alerts})
    slack = obs.get('slack_thread',[])

    query_user = f"INCIDENT: {obs.get('incident_title','')}\nALERTS:\n" + '\n'.join(f"[{a['timestamp']}] {a['service']}: {a['message']}" for a in alerts) + f"\nSLACK:\n" + '\n'.join(f"[{m['timestamp']}] {m['author']}: {m['text']}" for m in slack) + f"\nServices: {services}\nWhich to query?"
    qresp = call_llm(QUERY_SYSTEM, query_user)
    q = extract_json(qresp)
    result = env.step({'action_type':'QUERY_LOGS','query_service':(q or {}).get('service', services[0] if services else 'payments'),'query_from':(q or {}).get('from','00:00'),'query_to':(q or {}).get('to','23:59')})
    obs = result['observation']; logs = obs.get('retrieved_logs') or []
    pairs.append({'prompt_system':QUERY_SYSTEM,'prompt_user':query_user,'completion':qresp,'reward':float(result.get('reward',{}).get('total',0) or 0),'phase':'query'})

    ctx = build_context(obs, logs)
    for section in ['summary','root_cause']:
        wuser = f"{ctx}\n\nWRITE THE '{section.upper()}' SECTION:\n{SECTION_PROMPTS[section]}\n\nContent:"
        resp = call_llm(WRITE_SYSTEM, wuser)
        content = resp if resp and len(resp) >= 20 and not resp.startswith('{') else fallback_section(section, obs)
        result = env.step({'action_type':'WRITE_SECTION','section_name':section,'section_content':content})
        pairs.append({'prompt_system':WRITE_SYSTEM,'prompt_user':wuser,'completion':content,'reward':float(result.get('reward',{}).get('total',0) or 0),'phase':f'write_{section}'})
        obs = result['observation']

    result = env.step({'action_type':'REQUEST_REVIEW'})
    obs = result['observation']
    critiques = obs.get('skeptic_critiques',[])

    if critiques:
        critique = critiques[-1]
        current_rc = ''
        for s in obs.get('sections',[]):
            if s.get('name') == 'root_cause':
                current_rc = s.get('content','') or ''
        revise_user = f"INCIDENT: {obs.get('incident_title','')}\nORIGINAL ROOT_CAUSE: {current_rc}\nCRITIQUE: {critique}\nWrite revised root_cause addressing the critique:"
        revised = call_llm(REVISE_SYSTEM, revise_user)
        if not revised or len(revised) < 40:
            revised = f'REVISED: {current_rc} Addressing critique: {critique[:100]}'
        result = env.step({'action_type':'REVISE_SECTION','section_name':'root_cause','section_content':revised,'critique_addressed_index':len(critiques)-1})
        pairs.append({'prompt_system':REVISE_SYSTEM,'prompt_user':revise_user,'completion':revised,'reward':float(result.get('reward',{}).get('total',0) or 0),'phase':'revise_root_cause'})
        obs = result['observation']

    ctx = build_context(obs, logs)
    for section in ['timeline','impact','action_items']:
        wuser = f"{ctx}\n\nWRITE THE '{section.upper()}' SECTION:\n{SECTION_PROMPTS[section]}\n\nContent:"
        resp = call_llm(WRITE_SYSTEM, wuser)
        content = resp if resp and len(resp) >= 20 and not resp.startswith('{') else fallback_section(section, obs)
        result = env.step({'action_type':'WRITE_SECTION','section_name':section,'section_content':content})
        pairs.append({'prompt_system':WRITE_SYSTEM,'prompt_user':wuser,'completion':content,'reward':float(result.get('reward',{}).get('total',0) or 0),'phase':f'write_{section}'})
        obs = result['observation']

    authors = [m['author'] for m in slack if m.get('author') != 'pagerduty-bot']
    owner = authors[0] if authors else 'sre'
    env.step({'action_type':'ASSIGN_ACTION_ITEM','action_item_description':'Prevent recurrence','action_item_owner':owner,'action_item_due_date':'2024-08-15'})
    result = env.step({'action_type':'SUBMIT'})
    return pairs, result.get('info',{}).get('grade',{}).get('total_score',0)

print('Rollout functions defined.')

## Cell 6 — Collect teacher rollouts

In [ ]:
DIFFICULTIES = ['easy','medium','hard','expert']

all_pairs = []
episode_results = []
t_start = time.time()
total_episodes = N_ROLLOUTS_PER_DIFFICULTY * len(DIFFICULTIES)
ep_idx = 0

print(f'Collecting {total_episodes} episodes ({int(MULTI_AGENT_PROB*100)}% multi-agent)...\n')

for difficulty in DIFFICULTIES:
    for i in range(N_ROLLOUTS_PER_DIFFICULTY):
        ep_idx += 1
        is_multi = random.random() < MULTI_AGENT_PROB
        mode = 'MULTI' if is_multi else 'SINGLE'
        try:
            if is_multi:
                pairs, final = run_multi_agent_episode(env, difficulty)
            else:
                pairs, final = run_single_agent_episode(env, difficulty)
            for p in pairs:
                p['final_score'] = final
                p['difficulty']  = difficulty
                p['mode']        = mode
            all_pairs.extend(pairs)
            episode_results.append({'difficulty':difficulty,'mode':mode,'score':final,'pairs':len(pairs)})
            elapsed = time.time() - t_start
            print(f'[{ep_idx}/{total_episodes}] {difficulty:6s} {mode:5s} score={final:.3f} pairs={len(pairs)} elapsed={elapsed:.0f}s')
        except Exception as e:
            print(f'[{ep_idx}/{total_episodes}] {difficulty:6s} {mode:5s} ERROR: {e}')

print(f'\n=== Rollout collection complete ===')
print(f'  Total episodes: {len(episode_results)}')
print(f'  Total pairs:    {len(all_pairs)}')
print(f'  Time:           {(time.time()-t_start)/60:.1f} min')

df = pd.DataFrame(episode_results)
print('\nBy difficulty + mode:')
print(df.groupby(['difficulty','mode'])['score'].agg(['count','mean']))

## Cell 7 — Rejection sampling

In [ ]:
training_pairs = [p for p in all_pairs if p['final_score'] >= REWARD_THRESHOLD]

print(f'Before filtering: {len(all_pairs)} pairs')
print(f'After filtering (final_score >= {REWARD_THRESHOLD}): {len(training_pairs)} pairs')

from collections import Counter
print(f'\nMode distribution:  {dict(Counter(p["mode"] for p in training_pairs))}')
print(f'Phase distribution: {dict(Counter(p["phase"] for p in training_pairs))}')
print(f'New multi-agent phase pairs: {sum(1 for p in training_pairs if p["phase"]=="revise_root_cause")}')

assert len(training_pairs) >= 50, f'Only {len(training_pairs)} pairs - need more episodes.'

## Cell 8 — Load student model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f'Loading student: {STUDENT_MODEL}')
tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(STUDENT_MODEL, torch_dtype=torch.float16, device_map='cuda:0')
print(f'Loaded: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params on {model.device}')
print(f'GPU memory: {torch.cuda.memory_allocated()/(1024**3):.2f} GB')

## Cell 9 — Prepare training dataset

In [ ]:
from datasets import Dataset

def to_chat(p):
    return {
        'messages': [
            {'role':'system','content':p['prompt_system']},
            {'role':'user',   'content':p['prompt_user']},
            {'role':'assistant','content':p['completion']},
        ]
    }

train_data = [to_chat(p) for p in training_pairs]
train_dataset = Dataset.from_list(train_data)

print(f'Training dataset size: {len(train_dataset)}')

## Cell 10 — TRL SFT training

In [ ]:

import torch
import gc

# Aggressive memory cleanup
torch.cuda.empty_cache()
gc.collect()

print(f'GPU memory before: {torch.cuda.memory_allocated()/(1024**3):.2f} GB')
print(f'Model dtype: {next(model.parameters()).dtype}')

# Ensure bf16
if next(model.parameters()).dtype != torch.bfloat16:
    model = model.to(torch.bfloat16)
    torch.cuda.empty_cache()
    print(f'After bf16: {torch.cuda.memory_allocated()/(1024**3):.2f} GB')

from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir='/content/sft_v2_out',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    logging_steps=5,
    save_strategy='no',
    report_to='none',
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    max_seq_length=768,
    packing=False,
    dataloader_num_workers=0,
    optim='adamw_torch_fused',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=sft_config,
)

print('\nStarting training...')
trainer.train()
print('Training complete.')

## Cell 11 — Training loss chart

In [ ]:
log_hist = trainer.state.log_history
steps = [h.get('step',i) for i,h in enumerate(log_hist) if 'loss' in h]
losses = [h['loss'] for h in log_hist if 'loss' in h]

fig, ax = plt.subplots(figsize=(10,4))
ax.plot(steps, losses, marker='o', color='#3B82F6')
ax.set_xlabel('Training Step')
ax.set_ylabel('Training Loss')
ax.set_title('TRL SFT V2 Training Loss — Multi-Agent Enabled')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_loss_curve_v2.png', dpi=120)
plt.show()
print(f'Saved: {OUTPUT_DIR}/training_loss_curve_v2.png')
print(f'Final loss: {losses[-1]:.4f}')
print(f'Total steps: {len(losses)}')

## Cell 12 — Evaluate trained (V2) model

In [ ]:
def student_generate(messages, max_new_tokens=500):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def run_student_episode(env, difficulty, use_student=True):
    result = env.reset(difficulty=difficulty)
    obs = result['observation']
    alerts = obs.get('alerts',[])
    services = list({a['service'] for a in alerts})
    slack = obs.get('slack_thread',[])

    def llm_call(system, user):
        if use_student:
            return student_generate([{'role':'system','content':system},{'role':'user','content':user}])
        return call_llm(system, user)

    query_user = f"INCIDENT: {obs.get('incident_title','')}\nALERTS:\n" + '\n'.join(f"[{a['timestamp']}] {a['service']}: {a['message']}" for a in alerts) + f"\nSLACK:\n" + '\n'.join(f"[{m['timestamp']}] {m['author']}: {m['text']}" for m in slack) + f"\nServices: {services}\nWhich to query?"
    qresp = llm_call(QUERY_SYSTEM, query_user)
    q = extract_json(qresp)
    result = env.step({'action_type':'QUERY_LOGS','query_service':(q or {}).get('service', services[0] if services else 'payments'),'query_from':(q or {}).get('from','00:00'),'query_to':(q or {}).get('to','23:59')})
    obs = result['observation']; logs = obs.get('retrieved_logs') or []

    ctx = build_context(obs, logs)
    for section in ['summary','timeline','root_cause','impact','action_items']:
        wuser = f"{ctx}\n\nWRITE THE '{section.upper()}' SECTION:\n{SECTION_PROMPTS[section]}\n\nContent:"
        resp = llm_call(WRITE_SYSTEM, wuser)
        content = resp if resp and len(resp) >= 20 and not resp.startswith('{') else fallback_section(section, obs)
        result = env.step({'action_type':'WRITE_SECTION','section_name':section,'section_content':content})
        obs = result['observation']

    authors = [m['author'] for m in slack if m.get('author') != 'pagerduty-bot']
    owner = authors[0] if authors else 'sre'
    env.step({'action_type':'ASSIGN_ACTION_ITEM','action_item_description':'Prevent recurrence','action_item_owner':owner,'action_item_due_date':'2024-08-15'})
    result = env.step({'action_type':'SUBMIT'})
    return result.get('info',{}).get('grade',{}).get('total_score',0)

print('Evaluating TRAINED Qwen 0.5B (V2)...')
after_scores = {}
for d in DIFFICULTIES:
    try:
        s = run_student_episode(env, d, use_student=True)
        after_scores[d] = round(s, 4)
        print(f'  {d:6s}: {s:.3f}')
    except Exception as e:
        after_scores[d] = 0.0
        print(f'  {d:6s}: ERROR {e}')

print(f'\nAfter V2 training average: {sum(after_scores.values())/len(after_scores):.3f}')

## Cell 13 — Evaluate BASE (untrained) Qwen

In [ ]:
print('Loading BASE Qwen 0.5B (untrained)...')
base_model = AutoModelForCausalLM.from_pretrained(STUDENT_MODEL, torch_dtype=torch.float16, device_map='cuda:0')

_original_model = model
model = base_model

print('Evaluating BASE Qwen 0.5B...')
before_scores = {}
for d in DIFFICULTIES:
    try:
        s = run_student_episode(env, d, use_student=True)
        before_scores[d] = round(s, 4)
        print(f'  {d:6s}: {s:.3f}')
    except Exception as e:
        before_scores[d] = 0.0
        print(f'  {d:6s}: ERROR {e}')

model = _original_model
del base_model
torch.cuda.empty_cache()

print(f'\nBefore training average: {sum(before_scores.values())/len(before_scores):.3f}')

## Cell 14 — Results JSON + comparison chart

In [ ]:
before_avg = sum(before_scores.values()) / len(before_scores)
after_avg  = sum(after_scores.values())  / len(after_scores)
abs_imp    = after_avg - before_avg
rel_imp    = (abs_imp / before_avg * 100) if before_avg > 0 else 0

results = {
    'method':                 'TRL SFT V2 — Multi-Agent Enabled',
    'student_model':          STUDENT_MODEL,
    'teacher_model':          GROQ_MODEL,
    'training_examples':      len(training_pairs),
    'multi_agent_episodes':   sum(1 for r in episode_results if r['mode']=='MULTI'),
    'single_agent_episodes':  sum(1 for r in episode_results if r['mode']=='SINGLE'),
    'reward_threshold':       REWARD_THRESHOLD,
    'before_training':        before_scores,
    'after_training':         after_scores,
    'before_avg':             round(before_avg, 4),
    'after_avg':              round(after_avg,  4),
    'absolute_improvement':   round(abs_imp, 4),
    'relative_improvement_pct': round(rel_imp, 2),
    'final_training_loss':    round(losses[-1], 4) if losses else None,
}

with open(f'{OUTPUT_DIR}/training_results_v2.json','w') as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
x = list(DIFFICULTIES)
before_vals = [before_scores[d] for d in x]
after_vals  = [after_scores[d]  for d in x]

x_pos = np.arange(len(x))
w = 0.35
axes[0].bar(x_pos - w/2, before_vals, w, color='#94A3B8', label='Before V2')
axes[0].bar(x_pos + w/2, after_vals,  w, color='#10B981', label='After V2')
for i,(b,a) in enumerate(zip(before_vals, after_vals)):
    axes[0].text(i-w/2, b+0.02, f'{b:.2f}', ha='center', fontsize=10)
    axes[0].text(i+w/2, a+0.02, f'{a:.2f}', ha='center', fontsize=10, fontweight='bold')
axes[0].set_xticks(x_pos); axes[0].set_xticklabels([d.capitalize() for d in x])
axes[0].set_ylabel('Environment Reward Score')
axes[0].set_title('Reward Improvement per Task (V2)')
axes[0].set_ylim(0, 1.05); axes[0].legend(); axes[0].grid(alpha=0.3, axis='y')

axes[1].bar(['Before','After'], [before_avg, after_avg], color=['#94A3B8','#10B981'])
axes[1].text(0, before_avg+0.02, f'{before_avg:.3f}', ha='center', fontsize=12)
axes[1].text(1, after_avg+0.02,  f'{after_avg:.3f}',  ha='center', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Average Reward Score')
axes[1].set_title(f'Overall: {abs_imp:+.3f} ({rel_imp:+.1f}%)')
axes[1].set_ylim(0, 1.05); axes[1].grid(alpha=0.3, axis='y')

plt.suptitle('Qwen 2.5-0.5B V2 — TRL Fine-Tuned with Multi-Agent Episodes', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/reward_improvement_v2.png', dpi=120)
plt.show()
print(f'\nSaved chart: {OUTPUT_DIR}/reward_improvement_v2.png')

## Cell 15 — Save V2 model to disk + download instructions


In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f'V2 model saved to: {OUTPUT_DIR}')
print()
print('Files:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / (1024*1024)
    print(f'  {f:45s} {size_mb:>7.2f} MB')

print()
print('To download artifacts, use the Colab file browser (folder icon, left sidebar).')
print('Key files for the pitch:')
print(f'  {OUTPUT_DIR}/training_results_v2.json')
print(f'  {OUTPUT_DIR}/reward_improvement_v2.png')
print(f'  {OUTPUT_DIR}/training_loss_curve_v2.png')
